In [43]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

MINIO_ACCESS = "slavakoder"
MINIO_SECRET = "slavakoder"
DB_PASS = "airflow"

spark = SparkSession.builder \
    .appName('cleandata') \
    .config('spark.driver.memory', '2g') \
    .config('spark.executor.memory', '2g') \
    .config('spark.shuffle.partitions', '8') \
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.demo.type", "jdbc") \
    .config("spark.sql.catalog.demo.uri", "jdbc:postgresql://postgres:5432/airflow") \
    .config("spark.sql.catalog.demo.jdbc.user", "airflow") \
    .config("spark.sql.catalog.demo.jdbc.password", DB_PASS) \
    .config("spark.sql.catalog.demo.warehouse", "s3a://raw-bronze/warehouse") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.postgresql:postgresql:42.6.0") \
    .getOrCreate()
print('запускаемся')

spark.sparkContext.setLogLevel('WARN')

raw_data = 's3a://raw-bronze/landing/p2p_transfers/*.csv'

ddl_schema = "tx_id STRING, sender_id STRING, receiver_id STRING, amount STRING, currency STRING, status STRING, timestamp LONG"

df = spark.read.csv(raw_data, header=True, schema=ddl_schema)

запускаемся


In [44]:
df = (df
    .withColumn('timestamp', F.from_unixtime(F.col('timestamp')).cast('timestamp'))
    .withColumn('sender_id', F.regexp_replace(F.col('sender_id'), r'\s+', ''))
    .withColumn('receiver_id', F.regexp_replace(F.col('receiver_id'), r'\s+', ''))
    .withColumn('amount', F.regexp_replace(F.col('amount'), ',', '.').cast('double'))
    .withColumn('timestamp', F.coalesce(F.col('timestamp'), F.lit('1970-01-01 00:00:00')))
)
df = df.fillna('1970-01-01 00:00:00', subset=['timestamp'])
df = df.fillna({'status': 'Unknown'})

In [45]:
df.show(50, truncate=False)

+--------------------------------+---------+-----------+-------+--------+--------+-------------------+
|tx_id                           |sender_id|receiver_id|amount |currency|status  |timestamp          |
+--------------------------------+---------+-----------+-------+--------+--------+-------------------+
|387bca5885054d329c0db731d08a12b8|USR_918  |USR_24100  |4392.59|EUR     |SUCCESS |2026-05-01 00:19:31|
|e19664458a3c446d9c146931c2b2327c|USR_10344|USR_20866  |2991.76|KZT     |N/A     |2026-05-18 02:27:53|
|e700cfb352af4cfeafe5a748f6520297|USR_35863|USR_23320  |3555.1 |EUR     |N/A     |2026-04-30 03:54:41|
|615d0e48e39f4ca287a0ee3dcb9edd58|USR_47959|USR_4684   |1118.15|EUR     |N/A     |2026-05-08 03:53:45|
|cc2f47cc428840ec98090128f19cf3a5|USR_22377|USR_22810  |1560.27|KZT     |FAILED  |2026-04-25 14:17:49|
|4c0f1f2f8cd9438ba2294b153d1f2e63|USR_24923|USR_16954  |3334.37|KZT     |SUCCESS |2026-05-18 02:42:25|
|1d5d51916be2466d8cd6fab8195dd236|USR_49035|USR_30652  |1767.96|USD     |

In [9]:
df.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- sender_id: string (nullable = true)
 |-- receiver_id: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)

